# DepthWizard — RS3DAda Height Benchmark

**Question this notebook answers:** does RS3DAda (a purpose-built remote-sensing height model, MIT license, real pretrained weights) beat our shipped 5.91m RMSE baseline, measured on the SAME tiles in the SAME units?

**Why we're asking:** the paper reports 4.921m RMSE, but that figure is averaged across 6 datasets (only 2 of which are ours) in nDSM (above-ground) units. Our 5.91m is absolute-DSM RMSE on Jacksonville only. Those numbers are not comparable as published. This notebook makes them comparable: run RS3DAda on our exact JAX tiles, ground-reference its output the same way our own shadow-hybrid path is ground-referenced, and score with our own `validation.compute_metrics` (independently verified elsewhere against known injected error).

**No claim is made until this notebook's own output says so.**

---
### Before running
1. `Runtime → Change runtime type → T4 GPU`
2. Have `colab_subset.zip` already uploaded to Drive root (from the SAM benchmark step)
3. Upload `depthwizard_modules.zip` to Drive root too (bundles dfc2019_loader.py, validation.py, segmentation.py, terrain_classify.py, calibration/, rs3dada_eval.py)

## 1. Verify GPU

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU, then rerun.')

## 2. Mount Drive, unpack data + our modules

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!test -f /content/drive/MyDrive/colab_subset.zip || echo 'MISSING colab_subset.zip'
!test -f /content/drive/MyDrive/depthwizard_modules.zip || echo 'MISSING depthwizard_modules.zip'

!mkdir -p /content/colab_subset /content/dw_modules
!unzip -q -o /content/drive/MyDrive/colab_subset.zip -d /content/colab_subset
!unzip -q -o /content/drive/MyDrive/depthwizard_modules.zip -d /content/dw_modules

import sys
sys.path.insert(0, '/content/dw_modules')

!ls /content/colab_subset/rgb | wc -l
!ls /content/dw_modules

## 3. Clone SynRS3D and install its dependencies

In [ ]:
!git clone -q https://github.com/JTRNEO/SynRS3D /content/SynRS3d
%cd /content/SynRS3d
!pip -q install -r requirements.txt || echo 'requirements.txt not found or partial -- installing knowns below'
!pip -q install albumentations rasterio huggingface_hub
# GDAL: Colab's system GDAL via apt is more reliable than pip here
!apt -qq install -y gdal-bin python3-gdal > /dev/null
%cd /content

## 4. Download the RS3DAda height checkpoint

In [ ]:
import os
os.makedirs('/content/SynRS3d/pretrain', exist_ok=True)
from huggingface_hub import hf_hub_download
path = hf_hub_download(repo_id='JTRNEO/RS3DAda', filename='RS3DAda_vitl_DPT_height.pth',
                        local_dir='/content/SynRS3d/pretrain')
print('checkpoint at', path)
!ls -lh /content/SynRS3d/pretrain

## 5. Sanity check: inspect infer_height.py's real CLI before trusting the wrapper

This step exists because an earlier version of the wrapper script guessed the model's import path incorrectly. Read the actual script before running it at scale.

In [ ]:
!cat /content/SynRS3d/infer_height.py | head -60
!python /content/SynRS3d/infer_height.py --help

## 6. Single-tile test run

In [ ]:
import glob
tile0_rgb = sorted(glob.glob('/content/colab_subset/rgb/*_RGB.tif'))[0]
print('testing on', tile0_rgb)

!python /content/SynRS3d/infer_height.py \
  --data_dir {tile0_rgb} \
  --restore_from /content/SynRS3d/pretrain/RS3DAda_vitl_DPT_height.pth \
  --output_path /content/test_height.tif \
  --use_tta

import rasterio, numpy as np
with rasterio.open('/content/test_height.tif') as src:
    h = src.read(1)
print('output shape', h.shape, 'range', np.nanmin(h), np.nanmax(h))

## 7. Visual check before trusting any metric

This is the step that caught three bad results in the SAM benchmark. Do it here too.

In [ ]:
import matplotlib.pyplot as plt
import cv2

with rasterio.open(tile0_rgb) as src:
    rgb = np.transpose(src.read([1,2,3]), (1,2,0))
rgb_small = cv2.resize(rgb, (512,512))
h_small = cv2.resize(h, (512,512))

fig, ax = plt.subplots(1, 2, figsize=(14,7))
ax[0].imshow(rgb_small); ax[0].set_title('RGB'); ax[0].axis('off')
im = ax[1].imshow(h_small, cmap='terrain'); ax[1].set_title('RS3DAda predicted nDSM (m)'); ax[1].axis('off')
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

## 8. Full benchmark against our LiDAR ground truth

Runs `rs3dada_eval.py`, which calls `infer_height.py` per tile, ground-references the output, and scores with our own `validation.compute_metrics`.

In [ ]:
import os
os.environ['DW_DATA'] = '/content/colab_subset'
os.environ['SYNRS3D_DIR'] = '/content/SynRS3d'
os.environ['RS3DADA_CKPT'] = '/content/SynRS3d/pretrain/RS3DAda_vitl_DPT_height.pth'

import importlib, rs3dada_eval
importlib.reload(rs3dada_eval)
rs3dada_eval.run(max_tiles=20)

## 9. Save results back to Drive

In [ ]:
!cp /content/colab_subset/rs3dada_vs_baseline.json /content/drive/MyDrive/ 2>/dev/null && echo saved || echo 'run section 8 first'

---
## What to send back

1. The **RMSE/MAE line** from section 8's final summary
2. The **per-tile lines** printed during section 8
3. The **RGB vs predicted-height image** from section 7
4. Any **inference_failed** tiles and their error output

If RS3DAda beats 5.91m on this measurement, it's worth integrating as the height backbone (replacing Depth Anything V2, whose correlation with true height was measured at r≈0.08 on nadir imagery). If it doesn't, that stands and we look elsewhere rather than swapping on the paper's own aggregate number.